## 0. Создаем Spark Session

In [ ]:
import os
import warnings
from pathlib import Path

warnings.filterwarnings('ignore')

if 'JAVA_HOME' not in os.environ:
    candidate = '/Library/Java/JavaVirtualMachines/jdk-21.jdk/Contents/Home'
    if Path(candidate).exists():
        os.environ['JAVA_HOME'] = candidate

DATA_DIR = Path.cwd() / 'spark_demo_output'
DATA_DIR.mkdir(exist_ok=True)

In [ ]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName('spark-seminar-demo')
    .master('local[4]')
    .config('spark.sql.shuffle.partitions', 8)
    .config('spark.sql.repl.eagerEval.enabled', True)
    .config('spark.sql.execution.arrow.pyspark.enabled', True)
    .getOrCreate()
)

spark

## 1. Генерируем данные

Собираем синтетический датасет заказов и маленький датасет по пользователям.

In [ ]:
from pyspark.sql import functions as F

orders = (
    spark.range(0, 300_000)
    .withColumnRenamed('id', 'order_id')
    .withColumn('user_id', (F.col('order_id') % 50_000) + 1)
    .withColumn('product_id', (F.col('order_id') % 2_000) + 100)
    .withColumn('city', F.element_at(F.array(*[F.lit(x) for x in ['Moscow', 'Berlin', 'Tbilisi', 'Belgrade', 'Yerevan']]), (((F.col('order_id') % 5) + 1).cast('int'))))
    .withColumn('channel', F.element_at(F.array(*[F.lit(x) for x in ['web', 'mobile', 'partner']]), (((F.col('order_id') % 3) + 1).cast('int'))))
    .withColumn('status', F.element_at(F.array(*[F.lit(x) for x in ['new', 'paid', 'shipped', 'cancelled']]), (((F.col('order_id') % 4) + 1).cast('int'))))
    .withColumn('amount', F.round((F.rand(seed=42) * 1500) + 20, 2))
    .withColumn('discount', F.round(F.when(F.col('order_id') % 10 == 0, F.col('amount') * 0.15).otherwise(F.col('amount') * 0.03), 2))
    .withColumn('order_ts', F.expr("timestamp'2026-01-01 00:00:00' + interval 1 minute * order_id"))
    .withColumn('order_date', F.to_date('order_ts'))
)

users = (
    spark.range(1, 50_001)
    .withColumnRenamed('id', 'user_id')
    .withColumn('segment', F.element_at(F.array(*[F.lit(x) for x in ['B2C', 'B2B', 'VIP']]), (((F.col('user_id') % 3) + 1).cast('int'))))
    .withColumn('country', F.element_at(F.array(*[F.lit(x) for x in ['RU', 'DE', 'GE', 'RS', 'AM']]), (((F.col('user_id') % 5) + 1).cast('int'))))
    .withColumn('registration_date', F.expr("date'2024-01-01' + interval 1 day * (user_id % 365)"))
)


In [ ]:
type(users)

In [ ]:
orders.show()

In [ ]:
users.show()

In [ ]:
orders.printSchema()

In [ ]:
orders.count(), users.count()

## 1.1 Базовые команды

In [11]:
orders.select("order_id", "user_id", "city").show()

+--------+-------+--------+
|order_id|user_id|    city|
+--------+-------+--------+
|       0|      1|  Moscow|
|       1|      2|  Berlin|
|       2|      3| Tbilisi|
|       3|      4|Belgrade|
|       4|      5| Yerevan|
|       5|      6|  Moscow|
|       6|      7|  Berlin|
|       7|      8| Tbilisi|
|       8|      9|Belgrade|
|       9|     10| Yerevan|
|      10|     11|  Moscow|
|      11|     12|  Berlin|
|      12|     13| Tbilisi|
|      13|     14|Belgrade|
|      14|     15| Yerevan|
|      15|     16|  Moscow|
|      16|     17|  Berlin|
|      17|     18| Tbilisi|
|      18|     19|Belgrade|
|      19|     20| Yerevan|
+--------+-------+--------+
only showing top 20 rows


In [12]:
orders.filter(F.col('city') == 'Moscow').show()

+--------+-------+----------+------+-------+---------+-------+--------+-------------------+----------+
|order_id|user_id|product_id|  city|channel|   status| amount|discount|           order_ts|order_date|
+--------+-------+----------+------+-------+---------+-------+--------+-------------------+----------+
|       0|      1|       100|Moscow|    web|      new| 948.78|  142.32|2026-01-01 00:00:00|2026-01-01|
|       5|      6|       105|Moscow|partner|     paid| 795.99|   23.88|2026-01-01 00:05:00|2026-01-01|
|      10|     11|       110|Moscow| mobile|  shipped| 692.64|   103.9|2026-01-01 00:10:00|2026-01-01|
|      15|     16|       115|Moscow|    web|cancelled|1102.43|   33.07|2026-01-01 00:15:00|2026-01-01|
|      20|     21|       120|Moscow|partner|      new|  742.0|   111.3|2026-01-01 00:20:00|2026-01-01|
|      25|     26|       125|Moscow| mobile|     paid| 484.71|   14.54|2026-01-01 00:25:00|2026-01-01|
|      30|     31|       130|Moscow|    web|  shipped| 1153.3|  172.99|20

## 2. Lazy evaluation и action

In [14]:
%%time
lazy_pipeline = (
    orders
    .select('order_id', 'user_id', 'city', 'status', 'amount', 'discount', 'order_date')
    .withColumn('net_amount', F.round(F.col('amount') - F.col('discount'), 2))
    .filter(F.col('status') != 'cancelled')
)

CPU times: user 5.42 ms, sys: 4.58 ms, total: 10 ms
Wall time: 32.8 ms


In [16]:
lazy_pipeline.explain(mode='formatted')

== Physical Plan ==
* Project (5)
+- * Project (4)
   +- * Filter (3)
      +- * Project (2)
         +- * Range (1)


(1) Range [codegen id : 1]
Output [1]: [id#0L]
Arguments: Range (0, 300000, step=1, splits=Some(4))

(2) Project [codegen id : 1]
Output [5]: [id#0L AS order_id#1L, ((id#0L % 50000) + 1) AS user_id#2L, element_at([Moscow,Berlin,Tbilisi,Belgrade,Yerevan], cast(((id#0L % 5) + 1) as int), None, true) AS city#4, element_at([new,paid,shipped,cancelled], cast(((id#0L % 4) + 1) as int), None, true) AS status#6, round(((rand(42) * 1500.0) + 20.0), 2) AS amount#7]
Input [1]: [id#0L]

(3) Filter [codegen id : 1]
Input [5]: [order_id#1L, user_id#2L, city#4, status#6, amount#7]
Condition : NOT (status#6 = cancelled)

(4) Project [codegen id : 1]
Output [7]: [order_id#1L, user_id#2L, city#4, status#6, amount#7, round(CASE WHEN ((order_id#1L % 10) = 0) THEN (amount#7 * 0.15) ELSE (amount#7 * 0.03) END, 2) AS discount#8, cast(2026-01-01 00:00:00 + (INTERVAL '01' MINUTE * order_id#1L)

In [17]:
%%time
lazy_pipeline.show(20, truncate=False)

+--------+-------+--------+-------+-------+--------+----------+----------+
|order_id|user_id|city    |status |amount |discount|order_date|net_amount|
+--------+-------+--------+-------+-------+--------+----------+----------+
|0       |1      |Moscow  |new    |948.78 |142.32  |2026-01-01|806.46    |
|1       |2      |Berlin  |paid   |784.4  |23.53   |2026-01-01|760.87    |
|2       |3      |Tbilisi |shipped|1268.79|38.06   |2026-01-01|1230.73   |
|4       |5      |Yerevan |new    |1025.43|30.76   |2026-01-01|994.67    |
|5       |6      |Moscow  |paid   |795.99 |23.88   |2026-01-01|772.11    |
|6       |7      |Berlin  |shipped|1518.72|45.56   |2026-01-01|1473.16   |
|8       |9      |Belgrade|new    |1474.5 |44.24   |2026-01-01|1430.26   |
|9       |10     |Yerevan |paid   |1213.94|36.42   |2026-01-01|1177.52   |
|10      |11     |Moscow  |shipped|692.64 |103.9   |2026-01-01|588.74    |
|12      |13     |Tbilisi |new    |578.62 |17.36   |2026-01-01|561.26    |
|13      |14     |Belgrad

In [18]:
%%time
lazy_pipeline.count()

CPU times: user 1.15 ms, sys: 2.02 ms, total: 3.17 ms
Wall time: 94.5 ms


225000

In [ ]:
a = orders.select("*")
b = a.filter()
c = b.groupby()
d = c.filter()

## 2.1 Job, stage и shuffle сразу после lazy evaluation

Execution model:
- `action` создаёт `job`
- `job` состоит из `stages`
- `shuffle` создает новую `stage`. Чем больше `shuffle` необходимо исполнить, тем больше `stage` будет
- `tasks` обычно соответствуют числу `partitions`


In [19]:
base_df = (
    orders
    .select('order_id', 'city', 'channel', 'amount')
    .filter(F.col('amount') > 500)
    .withColumn('amount_with_tax', F.round(F.col('amount') * 1.2, 2))
)

base_df.explain()
base_df.rdd.getNumPartitions()

== Physical Plan ==
*(1) Project [order_id#1L, city#4, channel#5, amount#7, round((amount#7 * 1.2), 2) AS amount_with_tax#273]
+- *(1) Filter (isnotnull(amount#7) AND (amount#7 > 500.0))
   +- *(1) Project [id#0L AS order_id#1L, element_at([Moscow,Berlin,Tbilisi,Belgrade,Yerevan], cast(((id#0L % 5) + 1) as int), None, true) AS city#4, element_at([web,mobile,partner], cast(((id#0L % 3) + 1) as int), None, true) AS channel#5, round(((rand(42) * 1500.0) + 20.0), 2) AS amount#7]
      +- *(1) Range (0, 300000, step=1, splits=4)




4

In [20]:
base_df.count()

204159

In [21]:
shuffle_df = (
    base_df
    .groupBy('city')
    .agg(F.count('*').alias('orders_cnt'))
)

shuffle_df.explain('formatted')
shuffle_df.show()

== Physical Plan ==
AdaptiveSparkPlan (8)
+- HashAggregate (7)
   +- Exchange (6)
      +- HashAggregate (5)
         +- Project (4)
            +- Filter (3)
               +- Project (2)
                  +- Range (1)


(1) Range
Output [1]: [id#0L]
Arguments: Range (0, 300000, step=1, splits=Some(4))

(2) Project
Output [2]: [element_at([Moscow,Berlin,Tbilisi,Belgrade,Yerevan], cast(((id#0L % 5) + 1) as int), None, true) AS city#4, round(((rand(42) * 1500.0) + 20.0), 2) AS amount#7]
Input [1]: [id#0L]

(3) Filter
Input [2]: [city#4, amount#7]
Condition : (isnotnull(amount#7) AND (amount#7 > 500.0))

(4) Project
Output [1]: [city#4]
Input [2]: [city#4, amount#7]

(5) HashAggregate
Input [1]: [city#4]
Keys [1]: [city#4]
Functions [1]: [partial_count(1)]
Aggregate Attributes [1]: [count#290L]
Results [2]: [city#4, count#291L]

(6) Exchange
Input [2]: [city#4, count#291L]
Arguments: hashpartitioning(city#4, 8), ENSURE_REQUIREMENTS, [plan_id=389]

(7) HashAggregate
Input [2]: [city#4, co

In [22]:
repart_df = base_df.repartition(8, 'city')
repart_df.explain('formatted')
repart_df.count()

== Physical Plan ==
AdaptiveSparkPlan (6)
+- Exchange (5)
   +- Project (4)
      +- Filter (3)
         +- Project (2)
            +- Range (1)


(1) Range
Output [1]: [id#0L]
Arguments: Range (0, 300000, step=1, splits=Some(4))

(2) Project
Output [4]: [id#0L AS order_id#1L, element_at([Moscow,Berlin,Tbilisi,Belgrade,Yerevan], cast(((id#0L % 5) + 1) as int), None, true) AS city#4, element_at([web,mobile,partner], cast(((id#0L % 3) + 1) as int), None, true) AS channel#5, round(((rand(42) * 1500.0) + 20.0), 2) AS amount#7]
Input [1]: [id#0L]

(3) Filter
Input [4]: [order_id#1L, city#4, channel#5, amount#7]
Condition : (isnotnull(amount#7) AND (amount#7 > 500.0))

(4) Project
Output [5]: [order_id#1L, city#4, channel#5, amount#7, round((amount#7 * 1.2), 2) AS amount_with_tax#273]
Input [4]: [order_id#1L, city#4, channel#5, amount#7]

(5) Exchange
Input [5]: [order_id#1L, city#4, channel#5, amount#7, amount_with_tax#273]
Arguments: hashpartitioning(city#4, 8), REPARTITION_BY_NUM, [plan_i

204159

In [23]:
%%time
base_df_cached = base_df.repartition('city').cache()
base_df_cached.count()

CPU times: user 1.87 ms, sys: 2.04 ms, total: 3.91 ms
Wall time: 349 ms


204159

In [24]:
%%time
base_df_cached.count()

CPU times: user 1.02 ms, sys: 1.95 ms, total: 2.98 ms
Wall time: 57.8 ms


204159

In [25]:
%%time
base_df_persisted = base_df.repartition('city').persist()
base_df_persisted.count()

CPU times: user 2.16 ms, sys: 2.05 ms, total: 4.21 ms
Wall time: 60.9 ms


26/04/07 17:31:58 WARN CacheManager: Asked to cache already cached data.


204159

In [26]:
%%time
base_df_cached.count()

CPU times: user 802 μs, sys: 1.2 ms, total: 2.01 ms
Wall time: 58 ms


204159

In [27]:
%%time

from pyspark.storagelevel import StorageLevel

base_df_persisted_disk = base_df.repartition('city').persist(StorageLevel.DISK_ONLY_3)
base_df_persisted_disk.count()

CPU times: user 2.59 ms, sys: 2.63 ms, total: 5.22 ms
Wall time: 55.2 ms


26/04/07 17:32:39 WARN CacheManager: Asked to cache already cached data.


204159

## 3. Базовый DataFrame API

In [28]:
sales_df = (
    orders
    .select('order_id', 'user_id', 'city', 'channel', 'status', 'order_date', 'amount', 'discount')
    .withColumn('net_amount', F.round(F.col('amount') - F.col('discount'), 2))
    .withColumn(
        'amount_bucket',
        F.when(F.col('amount') < 100, 'small')
         .when(F.col('amount') < 500, 'medium')
         .when(F.col('amount') < 1000, 'large')
         .otherwise('xlarge')
    )
    .filter(F.col('status') != 'cancelled')
)

sales_df.show(10, truncate=False)

+--------+-------+--------+-------+-------+----------+-------+--------+----------+-------------+
|order_id|user_id|city    |channel|status |order_date|amount |discount|net_amount|amount_bucket|
+--------+-------+--------+-------+-------+----------+-------+--------+----------+-------------+
|0       |1      |Moscow  |web    |new    |2026-01-01|948.78 |142.32  |806.46    |large        |
|1       |2      |Berlin  |mobile |paid   |2026-01-01|784.4  |23.53   |760.87    |large        |
|2       |3      |Tbilisi |partner|shipped|2026-01-01|1268.79|38.06   |1230.73   |xlarge       |
|4       |5      |Yerevan |mobile |new    |2026-01-01|1025.43|30.76   |994.67    |xlarge       |
|5       |6      |Moscow  |partner|paid   |2026-01-01|795.99 |23.88   |772.11    |large        |
|6       |7      |Berlin  |web    |shipped|2026-01-01|1518.72|45.56   |1473.16   |xlarge       |
|8       |9      |Belgrade|partner|new    |2026-01-01|1474.5 |44.24   |1430.26   |xlarge       |
|9       |10     |Yerevan |web

In [29]:
sales_df.select('city', 'channel', 'amount', 'net_amount', 'amount_bucket').show(10, truncate=False)

+--------+-------+-------+----------+-------------+
|city    |channel|amount |net_amount|amount_bucket|
+--------+-------+-------+----------+-------------+
|Moscow  |web    |948.78 |806.46    |large        |
|Berlin  |mobile |784.4  |760.87    |large        |
|Tbilisi |partner|1268.79|1230.73   |xlarge       |
|Yerevan |mobile |1025.43|994.67    |xlarge       |
|Moscow  |partner|795.99 |772.11    |large        |
|Berlin  |web    |1518.72|1473.16   |xlarge       |
|Belgrade|partner|1474.5 |1430.26   |xlarge       |
|Yerevan |web    |1213.94|1177.52   |xlarge       |
|Moscow  |mobile |692.64 |588.74    |large        |
|Tbilisi |web    |578.62 |561.26    |large        |
+--------+-------+-------+----------+-------------+
only showing top 10 rows


## 4. GroupBy и агрегации

Считаем простые метрики  
`groupBy` часто приводит к `shuffle`


In [30]:
(
    sales_df
    .groupBy('city', 'channel')
    .agg(
        F.count('*').alias('orders_cnt'),
        F.round(F.sum('net_amount'), 2).alias('gmv'),
        F.round(F.avg('net_amount'), 2).alias('avg_check')
    )
    .orderBy(F.desc('gmv'))
)

city,channel,orders_cnt,gmv,avg_check
Berlin,partner,15000,1.132092719E7,754.73
Yerevan,partner,15000,1.127179371E7,751.45
Berlin,mobile,15000,1.122118839E7,748.08
Yerevan,mobile,15000,1.121502638E7,747.67
Belgrade,web,15000,1.120156136E7,746.77
Berlin,web,15000,1.119705891E7,746.47
Tbilisi,partner,15000,1.119064771E7,746.04
Belgrade,mobile,15000,1.117749443E7,745.17
Tbilisi,web,15000,1.116555902E7,744.37
Yerevan,web,15000,1.116520039E7,744.35


In [31]:
(
    sales_df
    .groupBy('city', 'channel')
    .agg(F.sum('net_amount').alias('revenue'))
    .explain(mode='formatted')
)


== Physical Plan ==
AdaptiveSparkPlan (8)
+- HashAggregate (7)
   +- Exchange (6)
      +- HashAggregate (5)
         +- Project (4)
            +- Filter (3)
               +- Project (2)
                  +- Range (1)


(1) Range
Output [1]: [id#0L]
Arguments: Range (0, 300000, step=1, splits=Some(4))

(2) Project
Output [5]: [id#0L AS order_id#1L, element_at([Moscow,Berlin,Tbilisi,Belgrade,Yerevan], cast(((id#0L % 5) + 1) as int), None, true) AS city#4, element_at([web,mobile,partner], cast(((id#0L % 3) + 1) as int), None, true) AS channel#5, element_at([new,paid,shipped,cancelled], cast(((id#0L % 4) + 1) as int), None, true) AS status#6, round(((rand(42) * 1500.0) + 20.0), 2) AS amount#7]
Input [1]: [id#0L]

(3) Filter
Input [5]: [order_id#1L, city#4, channel#5, status#6, amount#7]
Condition : NOT (status#6 = cancelled)

(4) Project
Output [3]: [city#4, channel#5, round((amount#7 - round(CASE WHEN ((order_id#1L % 10) = 0) THEN (amount#7 * 0.15) ELSE (amount#7 * 0.03) END, 2)), 2) A

## 5. Join и broadcast join

In [32]:
sales_df.count()

225000

In [33]:
users.count()

50000

In [38]:
joined = sales_df.join(users, on='user_id', how='left')
joined.select('user_id', 'segment', 'country', 'city', 'net_amount').show(10, truncate=False)

+-------+-------+-------+--------+----------+
|user_id|segment|country|city    |net_amount|
+-------+-------+-------+--------+----------+
|1      |B2B    |DE     |Moscow  |806.46    |
|2      |VIP    |GE     |Berlin  |760.87    |
|3      |B2C    |RS     |Tbilisi |1230.73   |
|5      |VIP    |RU     |Yerevan |994.67    |
|6      |B2C    |DE     |Moscow  |772.11    |
|7      |B2B    |GE     |Berlin  |1473.16   |
|9      |B2C    |AM     |Belgrade|1430.26   |
|10     |B2B    |RU     |Yerevan |1177.52   |
|11     |VIP    |DE     |Moscow  |588.74    |
|13     |B2B    |RS     |Tbilisi |561.26    |
+-------+-------+-------+--------+----------+
only showing top 10 rows


In [39]:
joined.explain(mode='formatted')

== Physical Plan ==
AdaptiveSparkPlan (11)
+- Project (10)
   +- BroadcastHashJoin LeftOuter BuildRight (9)
      :- Project (5)
      :  +- Project (4)
      :     +- Filter (3)
      :        +- Project (2)
      :           +- Range (1)
      +- BroadcastExchange (8)
         +- Project (7)
            +- Range (6)


(1) Range
Output [1]: [id#0L]
Arguments: Range (0, 300000, step=1, splits=Some(4))

(2) Project
Output [6]: [id#0L AS order_id#1L, ((id#0L % 50000) + 1) AS user_id#2L, element_at([Moscow,Berlin,Tbilisi,Belgrade,Yerevan], cast(((id#0L % 5) + 1) as int), None, true) AS city#4, element_at([web,mobile,partner], cast(((id#0L % 3) + 1) as int), None, true) AS channel#5, element_at([new,paid,shipped,cancelled], cast(((id#0L % 4) + 1) as int), None, true) AS status#6, round(((rand(42) * 1500.0) + 20.0), 2) AS amount#7]
Input [1]: [id#0L]

(3) Filter
Input [6]: [order_id#1L, user_id#2L, city#4, channel#5, status#6, amount#7]
Condition : NOT (status#6 = cancelled)

(4) Project
Ou

In [35]:
(
    joined
    .groupBy('segment', 'country')
    .agg(
        F.count('*').alias('orders_cnt'),
        F.round(F.sum('net_amount'), 2).alias('revenue')
    )
    .orderBy('segment', F.desc('revenue'))
).show()


+-------+-------+----------+-------------+
|segment|country|orders_cnt|      revenue|
+-------+-------+----------+-------------+
|    B2B|     GE|     15000|1.122716229E7|
|    B2B|     AM|     15000|1.119963552E7|
|    B2B|     RU|     15000|1.117429668E7|
|    B2B|     RS|     15000|1.109512736E7|
|    B2B|     DE|     15000|1.026924126E7|
|    B2C|     GE|     14994|1.127655248E7|
|    B2C|     RU|     15000|1.127404597E7|
|    B2C|     AM|     15000|1.124488686E7|
|    B2C|     RS|     15006| 1.12066278E7|
|    B2C|     DE|     15000|1.034945861E7|
|    VIP|     GE|     15006|1.123545972E7|
|    VIP|     RS|     14994|1.120764265E7|
|    VIP|     RU|     15000|1.120367783E7|
|    VIP|     AM|     15000|1.109880646E7|
|    VIP|     DE|     15000|1.030747997E7|
+-------+-------+----------+-------------+



In [36]:
broadcast_joined = sales_df.join(F.broadcast(users), on='user_id', how='inner')
broadcast_joined.explain(mode='formatted')

== Physical Plan ==
AdaptiveSparkPlan (11)
+- Project (10)
   +- BroadcastHashJoin Inner BuildRight (9)
      :- Project (5)
      :  +- Project (4)
      :     +- Filter (3)
      :        +- Project (2)
      :           +- Range (1)
      +- BroadcastExchange (8)
         +- Project (7)
            +- Range (6)


(1) Range
Output [1]: [id#0L]
Arguments: Range (0, 300000, step=1, splits=Some(4))

(2) Project
Output [6]: [id#0L AS order_id#1L, ((id#0L % 50000) + 1) AS user_id#2L, element_at([Moscow,Berlin,Tbilisi,Belgrade,Yerevan], cast(((id#0L % 5) + 1) as int), None, true) AS city#4, element_at([web,mobile,partner], cast(((id#0L % 3) + 1) as int), None, true) AS channel#5, element_at([new,paid,shipped,cancelled], cast(((id#0L % 4) + 1) as int), None, true) AS status#6, round(((rand(42) * 1500.0) + 20.0), 2) AS amount#7]
Input [1]: [id#0L]

(3) Filter
Input [6]: [order_id#1L, user_id#2L, city#4, channel#5, status#6, amount#7]
Condition : (NOT (status#6 = cancelled) AND isnotnull(user

## 6. Spark SQL

In [40]:
joined.createOrReplaceTempView('orders_joined')

In [41]:
spark.sql("""
SELECT city,
       segment,
       COUNT(*) AS orders_cnt,
       ROUND(SUM(net_amount), 2) AS revenue,
       ROUND(AVG(net_amount), 2) AS avg_order
  FROM orders_joined
 GROUP BY city, segment
 ORDER BY revenue DESC
 LIMIT 20
""").show()

+--------+-------+----------+-------------+---------+
|    city|segment|orders_cnt|      revenue|avg_order|
+--------+-------+----------+-------------+---------+
|  Berlin|    B2C|     14994|1.127655248E7|   752.07|
| Yerevan|    B2C|     15000|1.127404597E7|    751.6|
|Belgrade|    B2C|     15000|1.124488686E7|   749.66|
|  Berlin|    VIP|     15006|1.123545972E7|   748.73|
|  Berlin|    B2B|     15000|1.122716229E7|   748.48|
| Tbilisi|    VIP|     14994|1.120764265E7|   747.48|
| Tbilisi|    B2C|     15006| 1.12066278E7|   746.81|
| Yerevan|    VIP|     15000|1.120367783E7|   746.91|
|Belgrade|    B2B|     15000|1.119963552E7|   746.64|
| Yerevan|    B2B|     15000|1.117429668E7|   744.95|
|Belgrade|    VIP|     15000|1.109880646E7|   739.92|
| Tbilisi|    B2B|     15000|1.109512736E7|   739.68|
|  Moscow|    B2C|     15000|1.034945861E7|   689.96|
|  Moscow|    VIP|     15000|1.030747997E7|   687.17|
|  Moscow|    B2B|     15000|1.026924126E7|   684.62|
+--------+-------+----------

## 7. Оконные функции

In [42]:
from pyspark.sql.window import Window

city_window = Window.partitionBy('city').orderBy(F.desc('net_amount'))
daily_window = Window.partitionBy('city', 'order_date')

window_demo = (
    joined
    .withColumn('city_rank', F.dense_rank().over(city_window))
    .withColumn('daily_city_revenue', F.round(F.sum('net_amount').over(daily_window), 2))
    .select('city', 'order_date', 'order_id', 'net_amount', 'city_rank', 'daily_city_revenue')
)

window_demo.orderBy('city', 'city_rank').show(20, truncate=False)

+--------+----------+--------+----------+---------+------------------+
|city    |order_date|order_id|net_amount|city_rank|daily_city_revenue|
+--------+----------+--------+----------+---------+------------------+
|Belgrade|2026-05-16|194798  |1474.35   |1        |168734.75         |
|Belgrade|2026-01-01|288     |1474.34   |2        |157463.27         |
|Belgrade|2026-01-24|33148   |1474.32   |3        |157599.49         |
|Belgrade|2026-01-25|35108   |1474.3    |4        |163820.48         |
|Belgrade|2026-04-14|149338  |1474.29   |5        |157156.42         |
|Belgrade|2026-01-30|42628   |1474.26   |6        |148912.09         |
|Belgrade|2026-05-17|196108  |1474.26   |6        |172727.41         |
|Belgrade|2026-05-12|188768  |1474.25   |7        |162762.72         |
|Belgrade|2026-06-06|225968  |1474.23   |8        |162562.02         |
|Belgrade|2026-02-11|59818   |1474.12   |9        |157260.58         |
|Belgrade|2026-02-02|46373   |1474.11   |10       |168505.57         |
|Belgr

In [43]:
(
    window_demo
    .filter(F.col('city_rank') <= 3)
    .orderBy('city', 'city_rank', F.desc('net_amount'))
    .show(20, truncate=False)
)

+--------+----------+--------+----------+---------+------------------+
|city    |order_date|order_id|net_amount|city_rank|daily_city_revenue|
+--------+----------+--------+----------+---------+------------------+
|Belgrade|2026-05-16|194798  |1474.35   |1        |168734.75         |
|Belgrade|2026-01-01|288     |1474.34   |2        |157463.27         |
|Belgrade|2026-01-24|33148   |1474.32   |3        |157599.49         |
|Berlin  |2026-01-01|266     |1474.37   |1        |156154.2          |
|Berlin  |2026-07-27|298766  |1474.37   |1        |158566.35         |
|Berlin  |2026-01-28|39906   |1474.36   |2        |161716.45         |
|Berlin  |2026-07-21|290301  |1474.33   |3        |163661.31         |
|Moscow  |2026-01-20|28625   |1474.35   |1        |156875.15         |
|Moscow  |2026-05-18|198525  |1474.27   |2        |144506.64         |
|Moscow  |2026-01-22|30525   |1474.18   |3        |144279.52         |
|Tbilisi |2026-03-14|104182  |1474.4    |1        |159393.35         |
|Tbili

## 8. Cache, partitions и физический план

In [44]:
cached = joined.repartition('city').cache()
_ = cached.count()

In [45]:
cached.rdd.getNumPartitions(), orders.rdd.getNumPartitions()

(8, 4)

In [46]:
repartitioned = cached.repartition(8, 'city')
repartitioned.rdd.getNumPartitions()

8

In [47]:
(
    repartitioned
    .groupBy('city')
    .agg(F.sum('net_amount').alias('revenue'))
    .explain(mode='formatted')
)


== Physical Plan ==
AdaptiveSparkPlan (24)
+- HashAggregate (23)
   +- HashAggregate (22)
      +- InMemoryTableScan (1)
            +- InMemoryRelation (2)
                  +- AdaptiveSparkPlan (21)
                     +- == Final Plan ==
                        ResultQueryStage (16)
                        +- ShuffleQueryStage (15), Statistics(sizeInBytes=34.3 MiB, rowCount=2.25E+5)
                           +- Exchange (14)
                              +- * Project (13)
                                 +- * BroadcastHashJoin LeftOuter BuildRight (12)
                                    :- * Project (7)
                                    :  +- * Project (6)
                                    :     +- * Filter (5)
                                    :        +- * Project (4)
                                    :           +- * Range (3)
                                    +- BroadcastQueryStage (11), Statistics(sizeInBytes=4.4 MiB, rowCount=5.00E+4)
                             

In [48]:
(
    repartitioned
    .groupBy('city')
    .agg(F.sum('net_amount').alias('revenue'))
).show()

+--------+--------------------+
|    city|             revenue|
+--------+--------------------+
|Belgrade|3.3543328840000145E7|
|  Moscow| 3.092617983999994E7|
|  Berlin|3.3739174490000054E7|
| Yerevan|3.3652020480000235E7|
| Tbilisi|  3.35093978099999E7|
+--------+--------------------+



## 9. Pivot

In [ ]:
(
    joined
    .groupBy('city')
    .pivot('channel', ['web', 'mobile', 'partner'])
    .agg(F.round(F.sum('net_amount'), 2))
    .fillna(0)
    .orderBy('city')
)


## 10. Запись и чтение Parquet

In [54]:
output_path = DATA_DIR / 'orders_parquet'
(
    joined
    .repartition(20)
    .write
    .mode('overwrite')
    .partitionBy('city')
    .parquet(str(output_path))
)

In [55]:
!ls -lha /Users/avmysh/projs/2026/etl/notebooks/spark_demo_output/orders_parquet/city=Berlin

total 2728
drwxr-xr-x  42 avmysh  staff   1,3K  7 апр.  18:06 .
drwxr-xr-x   9 avmysh  staff   288B  7 апр.  18:06 ..
-rw-r--r--   1 avmysh  staff   512B  7 апр.  18:06 .part-00000-1c20a937-092a-4f92-a549-40120cc9d49c.c000.snappy.parquet.crc
-rw-r--r--   1 avmysh  staff   516B  7 апр.  18:06 .part-00001-1c20a937-092a-4f92-a549-40120cc9d49c.c000.snappy.parquet.crc
-rw-r--r--   1 avmysh  staff   504B  7 апр.  18:06 .part-00002-1c20a937-092a-4f92-a549-40120cc9d49c.c000.snappy.parquet.crc
-rw-r--r--   1 avmysh  staff   488B  7 апр.  18:06 .part-00003-1c20a937-092a-4f92-a549-40120cc9d49c.c000.snappy.parquet.crc
-rw-r--r--   1 avmysh  staff   508B  7 апр.  18:06 .part-00004-1c20a937-092a-4f92-a549-40120cc9d49c.c000.snappy.parquet.crc
-rw-r--r--   1 avmysh  staff   512B  7 апр.  18:06 .part-00005-1c20a937-092a-4f92-a549-40120cc9d49c.c000.snappy.parquet.crc
-rw-r--r--   1 avmysh  staff   508B  7 апр.  18:06 .part-00006-1c20a937-092a-4f92-a549-40120cc9d49c.c000.snappy.parquet.crc
-rw-r--r--   1

In [51]:
!ls -lha /Users/avmysh/projs/2026/etl/notebooks/spark_demo_output/orders_parquet

total 8
-rw-r--r--   1 avmysh  staff     0B  7 апр.  18:04 _SUCCESS
drwxr-xr-x   9 avmysh  staff   288B  7 апр.  18:04 .
-rw-r--r--   1 avmysh  staff     8B  7 апр.  18:04 ._SUCCESS.crc
drwxr-xr-x   4 avmysh  staff   128B  7 апр.  18:04 ..
drwxr-xr-x  10 avmysh  staff   320B  7 апр.  18:04 city=Belgrade
drwxr-xr-x  10 avmysh  staff   320B  7 апр.  18:04 city=Berlin
drwxr-xr-x  10 avmysh  staff   320B  7 апр.  18:04 city=Moscow
drwxr-xr-x  10 avmysh  staff   320B  7 апр.  18:04 city=Tbilisi
drwxr-xr-x  10 avmysh  staff   320B  7 апр.  18:04 city=Yerevan


In [50]:
output_path

PosixPath('/Users/avmysh/projs/2026/etl/notebooks/spark_demo_output/orders_parquet')

In [56]:
parquet_df = spark.read.parquet(str(output_path))
parquet_df.printSchema()
parquet_df.show(5, truncate=False)

root
 |-- user_id: long (nullable = true)
 |-- order_id: long (nullable = true)
 |-- channel: string (nullable = true)
 |-- status: string (nullable = true)
 |-- order_date: date (nullable = true)
 |-- amount: double (nullable = true)
 |-- discount: double (nullable = true)
 |-- net_amount: double (nullable = true)
 |-- amount_bucket: string (nullable = true)
 |-- segment: string (nullable = true)
 |-- country: string (nullable = true)
 |-- registration_date: timestamp (nullable = true)
 |-- city: string (nullable = true)

+-------+--------+-------+------+----------+------+--------+----------+-------------+-------+-------+-------------------+------+
|user_id|order_id|channel|status|order_date|amount|discount|net_amount|amount_bucket|segment|country|registration_date  |city  |
+-------+--------+-------+------+----------+------+--------+----------+-------------+-------+-------+-------------------+------+
|1717   |51716   |partner|new   |2026-02-05|990.58|29.72   |960.86    |large        

In [57]:
spark.read.parquet(str(output_path)).where(F.col('city') == 'Berlin').groupBy('segment').agg(F.count('*').alias('cnt')).show()

+-------+-----+
|segment|  cnt|
+-------+-----+
|    B2C|14994|
|    VIP|15006|
|    B2B|15000|
+-------+-----+



In [62]:
berlin_df = spark.read.parquet(str(output_path)).where(F.col('city') == 'Berlin').collect()

In [61]:
berlin_df['amount'].mean()

np.float64(772.9478968888889)

## 11. Pandas UDF

In [ ]:
import sys

In [66]:
# Import
from pyspark.sql import SparkSession

# Create SparkSession
spark = SparkSession.builder.appName('SparkByExamples.com') \
                    .getOrCreate()
# Prepare Data
columns = ["Seqno","Name"]
data = [("1", "john jones"),
    ("2", "tracey smith"),
    ("3", "amy sanders")]

# Create DataFrame
df = spark.createDataFrame(data=data,schema=columns)
df.show()

# imports
from pyspark.sql.functions import pandas_udf
from pyspark.sql.types import StringType
import pandas as pd

# create pandas_udf
@pandas_udf(StringType())
def to_upper(s: pd.Series) -> pd.Series:
    return s.str.upper()

# Using UDF with select()
df.select("Seqno","Name",to_upper("Name")).show()

# Using UDF with withColumn()
df.withColumn("upper_col",to_upper("Name")).show()

26/04/07 18:14:46 WARN SparkSession: Using an existing Spark session; only runtime SQL configurations will take effect.


+-----+------------+
|Seqno|        Name|
+-----+------------+
|    1|  john jones|
|    2|tracey smith|
|    3| amy sanders|
+-----+------------+



+-----+------------+--------------+
|Seqno|        Name|to_upper(Name)|
+-----+------------+--------------+
|    1|  john jones|    JOHN JONES|
|    2|tracey smith|  TRACEY SMITH|
|    3| amy sanders|   AMY SANDERS|
+-----+------------+--------------+

+-----+------------+------------+
|Seqno|        Name|   upper_col|
+-----+------------+------------+
|    1|  john jones|  JOHN JONES|
|    2|tracey smith|TRACEY SMITH|
|    3| amy sanders| AMY SANDERS|
+-----+------------+------------+



# ML

In [68]:
from pyspark.ml.linalg import Vectors
from pyspark.ml.stat import Correlation

data = [(Vectors.sparse(4, [(0, 1.0), (3, -2.0)]),),
        (Vectors.dense([4.0, 5.0, 0.0, 3.0]),),
        (Vectors.dense([6.0, 7.0, 0.0, 8.0]),),
        (Vectors.sparse(4, [(0, 9.0), (3, 1.0)]),)]
df = spark.createDataFrame(data, ["features"])

r1 = Correlation.corr(df, "features").head()


print("Pearson correlation matrix:\n" + str(r1[0]))

r2 = Correlation.corr(df, "features", "spearman").head()


print("Spearman correlation matrix:\n" + str(r2[0]))

Pearson correlation matrix:
DenseMatrix([[1.        , 0.05564149,        nan, 0.40047142],
             [0.05564149, 1.        ,        nan, 0.91359586],
             [       nan,        nan, 1.        ,        nan],
             [0.40047142, 0.91359586,        nan, 1.        ]])
Spearman correlation matrix:
DenseMatrix([[1.        , 0.10540926,        nan, 0.4       ],
             [0.10540926, 1.        ,        nan, 0.9486833 ],
             [       nan,        nan, 1.        ,        nan],
             [0.4       , 0.9486833 ,        nan, 1.        ]])


26/04/07 18:20:16 WARN PearsonCorrelation: Pearson correlation matrix contains NaN values.
26/04/07 18:20:16 WARN PearsonCorrelation: Pearson correlation matrix contains NaN values.


In [69]:
from pyspark.ml.linalg import Vectors
from pyspark.ml.stat import ChiSquareTest

data = [(0.0, Vectors.dense(0.5, 10.0)),
        (0.0, Vectors.dense(1.5, 20.0)),
        (1.0, Vectors.dense(1.5, 30.0)),
        (0.0, Vectors.dense(3.5, 30.0)),
        (0.0, Vectors.dense(3.5, 40.0)),
        (1.0, Vectors.dense(3.5, 40.0))]
df = spark.createDataFrame(data, ["label", "features"])

r = ChiSquareTest.test(df, "features", "label").head()



print("pValues: " + str(r.pValues))
print("degreesOfFreedom: " + str(r.degreesOfFreedom))
print("statistics: " + str(r.statistics))

pValues: [0.6872892787909721,0.6822703303362126]
degreesOfFreedom: [2, 3]
statistics: [0.75,1.5]


In [70]:
from pyspark.ml.classification import LogisticRegression

training = spark.read.format("libsvm").load("data/mllib/sample_libsvm_data.txt")

lr = LogisticRegression(maxIter=10, regParam=0.3, elasticNetParam=0.8)

lrModel = lr.fit(training)

print("Coefficients: " + str(lrModel.coefficients))
print("Intercept: " + str(lrModel.intercept))

mlr = LogisticRegression(maxIter=10, regParam=0.3, elasticNetParam=0.8, family="multinomial")

mlrModel = mlr.fit(training)

print("Multinomial coefficients: " + str(mlrModel.coefficientMatrix))
print("Multinomial intercepts: " + str(mlrModel.interceptVector))

26/04/07 18:20:30 WARN FileStreamSink: Assume no metadata directory. Error while looking for metadata directory in the path: data/mllib/sample_libsvm_data.txt.
java.io.FileNotFoundException: File data/mllib/sample_libsvm_data.txt does not exist
	at org.apache.hadoop.fs.RawLocalFileSystem.deprecatedGetFileStatus(RawLocalFileSystem.java:980)
	at org.apache.hadoop.fs.RawLocalFileSystem.getFileLinkStatusInternal(RawLocalFileSystem.java:1301)
	at org.apache.hadoop.fs.RawLocalFileSystem.getFileStatus(RawLocalFileSystem.java:970)
	at org.apache.hadoop.fs.FilterFileSystem.getFileStatus(FilterFileSystem.java:462)
	at org.apache.spark.sql.execution.streaming.sinks.FileStreamSink$.hasMetadata(FileStreamSink.scala:58)
	at org.apache.spark.sql.execution.datasources.DataSource.resolveRelation(DataSource.scala:384)
	at org.apache.spark.sql.catalyst.analysis.ResolveDataSource.org$apache$spark$sql$catalyst$analysis$ResolveDataSource$$loadV1BatchSource(ResolveDataSource.scala:143)
	at org.apache.spark.s

AnalysisException: [PATH_NOT_FOUND] Path does not exist: file:/Users/avmysh/projs/2026/etl/notebooks/data/mllib/sample_libsvm_data.txt. SQLSTATE: 42K03

In [71]:
from pyspark.ml import Pipeline
from pyspark.ml.classification import LogisticRegression
from pyspark.ml.evaluation import BinaryClassificationEvaluator
from pyspark.ml.feature import HashingTF, Tokenizer
from pyspark.ml.tuning import CrossValidator, ParamGridBuilder

# Prepare training documents, which are labeled.
training = spark.createDataFrame([
    (0, "a b c d e spark", 1.0),
    (1, "b d", 0.0),
    (2, "spark f g h", 1.0),
    (3, "hadoop mapreduce", 0.0),
    (4, "b spark who", 1.0),
    (5, "g d a y", 0.0),
    (6, "spark fly", 1.0),
    (7, "was mapreduce", 0.0),
    (8, "e spark program", 1.0),
    (9, "a e c l", 0.0),
    (10, "spark compile", 1.0),
    (11, "hadoop software", 0.0)
], ["id", "text", "label"])

# Configure an ML pipeline, which consists of tree stages: tokenizer, hashingTF, and lr.
tokenizer = Tokenizer(inputCol="text", outputCol="words")
hashingTF = HashingTF(inputCol=tokenizer.getOutputCol(), outputCol="features")
lr = LogisticRegression(maxIter=10)
pipeline = Pipeline(stages=[tokenizer, hashingTF, lr])

# We now treat the Pipeline as an Estimator, wrapping it in a CrossValidator instance.
# This will allow us to jointly choose parameters for all Pipeline stages.
# A CrossValidator requires an Estimator, a set of Estimator ParamMaps, and an Evaluator.
# We use a ParamGridBuilder to construct a grid of parameters to search over.
# With 3 values for hashingTF.numFeatures and 2 values for lr.regParam,
# this grid will have 3 x 2 = 6 parameter settings for CrossValidator to choose from.
paramGrid = ParamGridBuilder() \
    .addGrid(hashingTF.numFeatures, [10, 100, 1000]) \
    .addGrid(lr.regParam, [0.1, 0.01]) \
    .build()

crossval = CrossValidator(estimator=pipeline,
                          estimatorParamMaps=paramGrid,
                          evaluator=BinaryClassificationEvaluator(),
                          numFolds=2)  # use 3+ folds in practice

# Run cross-validation, and choose the best set of parameters.
cvModel = crossval.fit(training)

# Prepare test documents, which are unlabeled.
test = spark.createDataFrame([
    (4, "spark i j k"),
    (5, "l m n"),
    (6, "mapreduce spark"),
    (7, "apache hadoop")
], ["id", "text"])

# Make predictions on test documents. cvModel uses the best model found (lrModel).
prediction = cvModel.transform(test)
selected = prediction.select("id", "text", "probability", "prediction")
for row in selected.collect():
    print(row)

Row(id=4, text='spark i j k', probability=DenseVector([0.2665, 0.7335]), prediction=1.0)
Row(id=5, text='l m n', probability=DenseVector([0.9204, 0.0796]), prediction=0.0)
Row(id=6, text='mapreduce spark', probability=DenseVector([0.4438, 0.5562]), prediction=1.0)
Row(id=7, text='apache hadoop', probability=DenseVector([0.8587, 0.1413]), prediction=0.0)


## Хорошая практика -- остановить spark сессию

In [72]:
spark.stop()